In [2]:
cd DRAFT

/research/phd/phd2k22/cse/rudra.dhar/DRAFT


/research/phd/phd2k22/cse/rudra.dhar/miniconda3/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
pwd

'/research/phd/phd2k22/cse/rudra.dhar/DRAFT'

### Training

In [4]:
import os
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling, TrainerCallback
from datasets import load_dataset
from dotenv import load_dotenv
# from torch.utils.data import DataLoader
from copy import deepcopy

In [5]:
model_name = "google/gemma-3-4b-it"
cache_dir = "../cache"
output_dir = "DRAFT/Output/"

load_dotenv()
HUGGINGFACE_TOKEN = os.getenv("HUGGINGFACE")

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir, token=HUGGINGFACE_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=cache_dir,
    device_map="auto",
    dtype=torch.bfloat16,
    token=HUGGINGFACE_TOKEN
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
## For Context-Decision
# data_files = {"train": "Retrieval/CDtrain.jsonl", "validation": "Retrieval/CDval.jsonl"}
# dataset = load_dataset("json", data_files=data_files)

# ## For Title-Body
data_files = {"train": "Retrieval/TBtrain.jsonl", "validation": "Retrieval/TBval.jsonl"}
dataset = load_dataset("json", data_files=data_files)

In [8]:
dataset['train'] = dataset['train'].select(range(100))  # Use a smaller subset for training
dataset['validation'] = dataset['validation'].select(range(20))  # Use a smaller subset for validation

In [ ]:
# ## For Context-Decision
# def format_example(example):
#     retrieved_contexts = [doc["Context"] for doc in example["Retrieved"]]
#     retrieved_decisions = [doc["Decision"] for doc in example["Retrieved"]]
#     context = example["Anchor"]["Context"]
#     decision = example["Anchor"]["Decision"]

#     messages = [
#         {"role": "system", "content": "You are an expert software architect responsible for maintaining and thoroughly documenting all architectural decisions. You are writing an Architectural Decision Record for a software. Below are a few examples of Context and the corresponding Decision. Following the examples, provide only the ## Decision for the final ## Context provided by the user. Provide only the Decision in about 2-400 words. Do not add any explanations, introductions, or additional responses."},
#         {"role": "user", "content": f"## Context: {retrieved_contexts[0]}"},
#         {"role": "assistant", "content": f"## Decision: {retrieved_decisions[0]}"},
#         {"role": "user", "content": f"## Context: {retrieved_contexts[1]}"},
#         {"role": "assistant", "content": f"## Decision: {retrieved_decisions[1]}"},
#         {"role": "user", "content": f"## Context: {context}"},
#         {"role": "assistant", "content": f"## Decision: {decision}"}
#     ]
#     text = tokenizer.apply_chat_template(messages, tokenize=False)
#     return {"text": text}



## For Title-Body
def format_example(example):
    retrieved_title = [doc["Title"] for doc in example["Retrieved"]]
    retrieved_body = [doc["Body"] for doc in example["Retrieved"]]
    title = example["Anchor"]["Title"]
    body = example["Anchor"]["Body"]
    
    title = example["Anchor"]["Title"]
    body = example["Anchor"]["Body"]

    # Turn into chat-like input
    messages = [
        {"role": "system", "content": "You are an expert software architect responsible for maintaining and thoroughly documenting all architectural decisions. You are writing an Architectural Decision Record for a software. Write the ADR corresponding to the ADR Title provided by the User. Provide only the ADR content in about 10-800 words. Do not add any additional responses—only the ADR content."},
        {"role": "user", "content": f"# Title: {retrieved_title[0]}"},
        {"role": "assistant", "content": retrieved_body[0]},
        {"role": "user", "content": f"# Title: {retrieved_title[1]}"},
        {"role": "assistant", "content": retrieved_body[1]},
        {"role": "user", "content": f"# Title: {title}"},
        {"role": "assistant", "content": body}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

# the formated text is stored in the "text" field
dataset = dataset.map(format_example)


In [ ]:
# Tokenize
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=3200
    )

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

In [11]:
# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Training arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_strategy="epoch",
    # learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    # warmup_steps=20,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=False,
    bf16=True,
    save_total_limit=2,
    report_to="none",
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator
)

In [ ]:
# change path according to experiment
loss_log_path = "DRAFT/Output/loss_log_gemma-3-4b-it_TB.jsonl"

class CustomCallback(TrainerCallback):
    
    def __init__(self, trainer) -> None:
        super().__init__()
        self._trainer = trainer
    
    def on_epoch_end(self, args, state, control, **kwargs):
        # if control.should_evaluate:
        control_copy = deepcopy(control)
        train_metrics = self._trainer.evaluate(eval_dataset=self._trainer.train_dataset, metric_key_prefix="train_epoch")
        val_metrics = self._trainer.evaluate(eval_dataset=self._trainer.eval_dataset, metric_key_prefix="val_epoch")
        
        # Print the metrics instead of logging to WandB
        print(f"Training set evaluation at epoch {state.epoch}:")
        for key, value in train_metrics.items():
            print(f"  {key}: {value}")
        
        print(f"Validation set evaluation at epoch {state.epoch}:")
        for key, value in val_metrics.items():
            print(f"  {key}: {value}")
        
        log_data = {"epoch": state.epoch, "train_loss": train_metrics['train_epoch_loss'], "val_loss": val_metrics['val_epoch_loss']}
        with open(loss_log_path, 'a') as f:
            f.write(json.dumps(log_data) + '\n')
        
        return control_copy
    

trainer.add_callback(CustomCallback(trainer))


In [13]:
# Evaluate on training set
train_eval_results = trainer.evaluate(eval_dataset=trainer.train_dataset, metric_key_prefix="train")
train_loss = train_eval_results["train_loss"]

# Evaluate on validation set
val_eval_results = trainer.evaluate()
val_loss = val_eval_results["eval_loss"]

# Print
print(f"Epoch 0 - Train loss: {train_loss:.4f}, Validation loss: {val_loss:.4f}")
log_data = {"epoch": 0, "train_loss": train_loss, "val_loss": val_loss}

with open(loss_log_path, 'w') as f:
    f.write(json.dumps(log_data) + '\n')

Epoch 0 - Train loss: 4.4851, Validation loss: 4.4357


In [14]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss,Epoch Loss,Epoch Model Preparation Time,Model Preparation Time
1,21.468400,2.241048,2.241048,0.016100,0.016100
2,11.211200,2.284534,2.284534,0.016100,0.016100
3,7.008000,2.450914,2.450914,0.016100,0.016100


Training set evaluation at epoch 1.0:
  train_epoch_loss: 1.7058082818984985
  train_epoch_model_preparation_time: 0.0161
Validation set evaluation at epoch 1.0:
  val_epoch_loss: 2.2410476207733154
  val_epoch_model_preparation_time: 0.0161
Training set evaluation at epoch 2.0:
  train_epoch_loss: 1.0785541534423828
  train_epoch_model_preparation_time: 0.0161
Validation set evaluation at epoch 2.0:
  val_epoch_loss: 2.284534215927124
  val_epoch_model_preparation_time: 0.0161
Training set evaluation at epoch 3.0:
  train_epoch_loss: 0.8763749003410339
  train_epoch_model_preparation_time: 0.0161
Validation set evaluation at epoch 3.0:
  val_epoch_loss: 2.450913667678833
  val_epoch_model_preparation_time: 0.0161


TrainOutput(global_step=39, training_loss=13.229197771121294, metrics={'train_runtime': 938.9087, 'train_samples_per_second': 0.32, 'train_steps_per_second': 0.042, 'total_flos': 2.003986980864e+16, 'train_loss': 13.229197771121294, 'epoch': 3.0})

### Inference

In [30]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
import time

In [ ]:
checkpoint_dir = "DRAFT/Output/checkpoint-13"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
model = AutoModelForCausalLM.from_pretrained(
    checkpoint_dir,
    device_map="auto",
    dtype=torch.bfloat16
)

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move the model to the chosen device
model.to(device)

# Set the model to evaluation mode
model.eval()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using device: cuda


Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(4096, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (self_attn): SiglipAttention(
                (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
              )
              (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwi

In [ ]:
def load_jsonl(file_path):
    """Load a JSONL file and return list of dicts."""
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
            
        
    return data

def save_jsonl(data, file_path):
    """Append list of dicts to a JSONL file."""
    with open(file_path, "a", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
        
    

def generate_response(model, tokenizer, messages, device):
    """Generate model response given messages."""
    formatted_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Tokenize
    inputs = tokenizer(formatted_chat, return_tensors="pt").to(device)
    input_length = inputs["input_ids"].shape[1]

    # max_new_tokens=500 for Context-Decision; and 1000 for Title-Body
    outputs = model.generate(**inputs, max_new_tokens=1000)

    generated_ids = outputs[0][input_length:]  # slice only new tokens
    response = tokenizer.decode(generated_ids, skip_special_tokens=True) # Decode only the generated text
    generated_tokens = generated_ids.shape[0] # Number of generated tokens
    return response, generated_tokens


def extract_context(entry):
    """Extract context string from one JSONL entry."""
    return entry["Anchor"]["Context"]

def extract_retrieved_context(entry):
    """Extract retrieved context string from one JSONL entry."""
    contexts = [doc["Context"] for doc in entry["Retrieved"]]
    return contexts

def extract_retrieved_decision(entry):
    """Extract retrieved decision string from one JSONL entry."""
    decisions = [doc["Decision"] for doc in entry["Retrieved"]]
    return decisions

def extract_title(entry):
    """Extract title from one JSONL entry."""
    return entry["Anchor"]["Title"]

def extract_retrieved_title(entry):
    """Extract retrieved title from one JSONL entry."""
    titles = [doc["Title"] for doc in entry["Retrieved"]]
    return titles

def extract_retrieved_body(entry):
    """Extract retrieved body from one JSONL entry."""
    bodies = [doc["Body"] for doc in entry["Retrieved"]]
    return bodies

def extract_primary_key(entry):
    """Extract primary key from one JSONL entry."""
    return entry["Anchor"]["PrimaryKey"]

def context_formator(retrieved_contexts, retrieved_decisions, context):
    messages = [
        {"role": "system", "content": "You are an expert software architect responsible for maintaining and thoroughly documenting all architectural decisions. You are writing an Architectural Decision Record for a software. Below are a few examples of Context and the corresponding Decision. Following the examples, provide only the ## Decision for the final ## Context provided by the user. Provide only the Decision in about 2-400 words. Do not add any explanations, introductions, or additional responses."},
        {"role": "user", "content": f"## Context: {retrieved_contexts[0]}"},
        {"role": "assistant", "content": f"## Decision: {retrieved_decisions[0]}"},
        {"role": "user", "content": f"## Context: {retrieved_contexts[1]}"},
        {"role": "assistant", "content": f"## Decision: {retrieved_decisions[1]}"},
        {"role": "user", "content": f"## Context: {context}"},
    ]
    return messages

def title_formator(retrieved_title, retrieved_body, title):
    messages = [
        {"role": "system", "content": "You are an expert software architect responsible for maintaining and thoroughly documenting all architectural decisions. You are writing an Architectural Decision Record for a software. Write the ADR corresponding to the ADR Title provided by the User. Provide only the ADR content in about 10-800 words. Do not add any additional responses—only the ADR content."},
        {"role": "user", "content": f"# {retrieved_title[0]}"},
        {"role": "assistant", "content": retrieved_body[0]},
        {"role": "user", "content": f"# {retrieved_title[1]}"},
        {"role": "assistant", "content": retrieved_body[1]},
        {"role": "user", "content": f"# {title}"},
    ]
    return messages


In [ ]:
# This is for Context-Decision

# input_file = "Retrieval/CDtest.jsonl"
# output_file = "DRAFT/Results/gemma-3-4b-it-CDtest.jsonl"
# entries = load_jsonl(input_file)

# results = []

# # Iterate over entries
# for i, entry in enumerate(entries[:3]): # limit to first 3 for demo
#     primary_key = extract_primary_key(entry)
#     context = extract_context(entry)
#     retrieved_contexts = extract_retrieved_context(entry)
#     retrieved_decisions = extract_retrieved_decision(entry)
#     messages = context_formator(retrieved_contexts, retrieved_decisions, context)

#     start_time = time.time()
#     response, gen_tokens = generate_response(model, tokenizer, messages, device)
#     elapsed = time.time() - start_time


#     result = {
#     "PrimaryKey": primary_key,
#     "Decision": response,
#     "GeneratedTokens": gen_tokens,
#     "Time": elapsed
#     }
#     results.append(result)

# # Save all results to output JSONL
# save_jsonl(results, output_file)
# print(f"Results saved to {output_file}")

In [ ]:
# ## This is for Title-Body

input_file = "Retrieval/TBtest.jsonl"
output_file = "FinetDRAFTune/Results/gemma-3-4b-it-TBtest.jsonl"
entries = load_jsonl(input_file)

results = []

# Iterate over entries
for i, entry in enumerate(entries[:3]): # limit to first 3 for demo
    primary_key = extract_primary_key(entry)
    title = extract_title(entry)
    retrieved_title = extract_retrieved_title(entry)
    retrieved_body = extract_retrieved_body(entry)
    messages = context_formator(retrieved_title, retrieved_body, title)

    start_time = time.time()
    response, gen_tokens = generate_response(model, tokenizer, messages, device)
    elapsed = time.time() - start_time


    result = {
    "PrimaryKey": primary_key,
    "Body": response,
    "GeneratedTokens": gen_tokens,
    "Time": elapsed
    }
    results.append(result)

# Save all results to output JSONL
save_jsonl(results, output_file)
print(f"Results saved to {output_file}")

Results saved to Finetune/Results/gemma-3-4b-it-TBtest-results.jsonl
